In [ ]:
 pip install pandas numpy scikit-learn

In [ ]:
import re                                                          # Text cleaning via regex
import pandas as pd                                               # Data manipulation
from io import StringIO                                           # Read inline CSV string

from sklearn.model_selection import train_test_split              # Dataset splitting
from sklearn.feature_extraction.text import TfidfVectorizer       # TF-IDF features
from sklearn.naive_bayes import MultinomialNB                     # ML algorithm
from sklearn.metrics import (                                     # Evaluation metrics
    accuracy_score,
    confusion_matrix,
    classification_report,
)


In [ ]:
# Pipe (|) is used as the delimiter to avoid conflicts with commas in messages.
# To use your own CSV file, replace load_data() with:
#   df = pd.read_csv("your_file.csv")   # must have "Message" and "Label" columns

SAMPLE_DATA = """\
Message|Label
Congratulations you have won a 1000 dollar gift card click here to claim now|Spam
Hey are we still meeting for lunch tomorrow|Ham
URGENT your account has been compromised verify now at fakebank dot com|Spam
Can you please send me the project report by end of day|Ham
FREE iPhone limited time offer reply WIN to claim your prize|Spam
Don't forget about the team meeting at 3 PM today|Ham
You have been selected for a special reward call 1-800-FAKE now|Spam
Hi just checking in on how you are doing this week|Ham
Win a luxury cruise enter your details to claim this exclusive offer|Spam
Reminder your dentist appointment is scheduled for Friday at 10 AM|Ham
Earn 5000 dollars a week from home no experience needed sign up today|Spam
Could you review my code and let me know if there are any issues|Ham
ALERT suspicious login detected click to secure your account immediately|Spam
The quarterly financial report is attached please review before our call|Ham
Buy cheap medications online no prescription required lowest prices|Spam
I wanted to thank you for your help with the presentation yesterday|Ham
Your loan has been pre-approved get 10000 dollars instantly apply now|Spam
Let us catch up over coffee sometime this week if you are free|Ham
You are the lucky winner of our monthly sweepstakes claim now|Spam
Please find attached the minutes from last week meeting|Ham
Get rich quick invest 100 dollars today and earn 10000 tomorrow|Spam
Are you available for a quick call this afternoon to discuss the project|Ham
Special discount offer just for you buy now and save 80 percent|Spam
I have updated the spreadsheet with the latest sales figures|Ham
Your credit card has been charged if not you click here immediately|Spam
Looking forward to seeing you at the conference next week|Ham
Make money fast no skills required work from home starting today|Spam
Can you pick up the kids from school today I will be stuck in a meeting|Ham
Exclusive deal for you only cheap luxury watches buy now|Spam
The proposal looks great just a few minor edits and we are good to go|Ham
Lose 30 pounds in 30 days with this miracle pill order now|Spam
Just wanted to share a funny video I came across hope it makes you smile|Ham
Your package could not be delivered click here to reschedule|Spam
I am running a bit late to the office will be there in 20 minutes|Ham
You have been chosen to test our new product for FREE register today|Spam
Happy birthday hope your day is filled with joy and laughter|Ham
Act now and get 3 months free limited offer expires tonight|Spam
I really enjoyed the book you recommended finished it over the weekend|Ham
Low cost life insurance get coverage for just 1 dollar a day click here|Spam
The client loved the presentation great work everyone|Ham
Dear winner you have been randomly selected for a cash prize of 5000|Spam
Please review the attached contract before our meeting on Thursday|Ham
Claim your free vacation package now limited spots available|Spam
Just a heads up I will be out of office next Monday and Tuesday|Ham
You owe taxes act now to avoid legal action call this number immediately|Spam
Thank you for the birthday wishes it really made my day|Ham
Increase your credit score overnight with this secret method|Spam
The updated design mockups are ready for your review|Ham
Hot singles in your area want to meet you click here now|Spam
See you at the quarterly review meeting this Friday at noon|Ham
"""

In [ ]:
def load_data() -> pd.DataFrame:
    """
    Loads the spam/ham dataset from the built-in sample string.
    Uses '|' as delimiter to safely handle commas inside messages.
    """
    df = pd.read_csv(StringIO(SAMPLE_DATA), sep="|")
    df.columns = df.columns.str.strip()          # Remove accidental whitespace

    print("=" * 60)
    print("          EMAIL SPAM CLASSIFIER")
    print("=" * 60)
    print(f"\n[✔] Dataset loaded  —  {len(df)} records")
    print(f"\nFirst 5 rows:")
    print(df.head().to_string(index=False))
    print(f"\nLabel distribution:\n{df['Label'].value_counts().to_string()}")
    return df


In [ ]:
def preprocess_text(text: str) -> str:
    """
    Cleans a single email message:
      1. Lowercase conversion
      2. Remove URLs
      3. Remove punctuation / special characters  (keep letters & spaces)
      4. Collapse multiple spaces to one
    """
    text = text.lower()                                  # Step 1 – lowercase
    text = re.sub(r"http\S+|www\S+", "", text)           # Step 2 – strip URLs
    text = re.sub(r"[^a-z\s]", "", text)                 # Step 3 – remove non-letters
    text = re.sub(r"\s+", " ", text).strip()             # Step 4 – clean whitespace
    return text


def preprocess_dataset(df: pd.DataFrame) -> pd.DataFrame:
    """Applies preprocess_text() to every message in the dataset."""
    df = df.copy()
    df["Cleaned_Message"] = df["Message"].apply(preprocess_text)
    print("\n[✔] Text preprocessing complete.")
    print("\nSample — original vs cleaned:")
    for _, row in df[["Message", "Cleaned_Message"]].head(3).iterrows():
        print(f"  Original : {row['Message'][:60]}")
        print(f"  Cleaned  : {row['Cleaned_Message'][:60]}\n")
    return df

In [ ]:
def encode_labels(df: pd.DataFrame):
    """
    Maps string labels to integers:
      Ham  → 0
      Spam → 1
    Returns feature Series X and integer label array y.
    """
    label_map = {"Ham": 0, "Spam": 1}
    X = df["Cleaned_Message"]
    y = df["Label"].map(label_map)
    print("[✔] Labels encoded  →  Ham = 0  |  Spam = 1")
    return X, y

In [ ]:
def split_dataset(X, y):
    """
    Splits data into 80 % training and 20 % testing sets.
    stratify=y ensures both sets keep the same Spam/Ham ratio.
    """
    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=0.2,
        random_state=42,
        stratify=y,
    )
    print(f"[✔] Dataset split   →  Train: {len(X_train)}  |  Test: {len(X_test)}")
    return X_train, X_test, y_train, y_test

In [ ]:
def vectorize_text(X_train, X_test):
    """
    Converts text into numerical TF-IDF feature matrices.

    TF-IDF (Term Frequency – Inverse Document Frequency):
      • TF  — how often a word appears in a document
      • IDF — how rare the word is across all documents
    Words that are common everywhere (e.g. 'the') get low weight;
    distinctive words (e.g. 'prize', 'claim') get higher weight.
    """
    vectorizer = TfidfVectorizer(stop_words="english", max_features=5000)
    X_train_tfidf = vectorizer.fit_transform(X_train)   # Fit + transform training set
    X_test_tfidf  = vectorizer.transform(X_test)        # Transform test set only

    print(f"[✔] TF-IDF done     →  Train matrix: {X_train_tfidf.shape}"
          f"  |  Test matrix: {X_test_tfidf.shape}")
    return vectorizer, X_train_tfidf, X_test_tfidf

In [ ]:
def train_model(X_train_tfidf, y_train) -> MultinomialNB:
    """
    Trains a Multinomial Naive Bayes classifier on the TF-IDF features.
    Naive Bayes is fast, lightweight, and performs very well on text data.
    """
    model = MultinomialNB()
    model.fit(X_train_tfidf, y_train)
    print("[✔] Model trained   →  Multinomial Naive Bayes")
    return model

In [ ]:

def evaluate_model(model, X_test_tfidf, y_test) -> None:
    """
    Prints three evaluation reports:
      1. Accuracy Score   — overall correct predictions
      2. Confusion Matrix — breakdown of TP / TN / FP / FN
      3. Classification Report — precision, recall, F1 per class
    """
    y_pred = model.predict(X_test_tfidf)

    accuracy = accuracy_score(y_test, y_pred)
    cm       = confusion_matrix(y_test, y_pred)
    report   = classification_report(y_test, y_pred, target_names=["Ham", "Spam"])

    print("\n" + "=" * 60)
    print("             MODEL EVALUATION RESULTS")
    print("=" * 60)

    print(f"\n  Accuracy Score : {accuracy * 100:.2f}%")

    print("\n  Confusion Matrix:")
    print("  (Rows = Actual label | Columns = Predicted label)\n")
    print(f"                   Predicted Ham    Predicted Spam")
    print(f"  Actual Ham            {cm[0][0]:<16}  {cm[0][1]}")
    print(f"  Actual Spam           {cm[1][0]:<16}  {cm[1][1]}")

    print("\n  Classification Report:")
    for line in report.splitlines():
        print("  " + line)


In [ ]:
def predict_message(model, vectorizer, raw_message: str) -> str:
    """
    Cleans and classifies a single raw email message.
    Returns 'Spam' or 'Ham (Not Spam)'.
    """
    cleaned  = preprocess_text(raw_message)
    features = vectorizer.transform([cleaned])
    label    = model.predict(features)[0]
    return "Spam" if label == 1 else "Ham (Not Spam)"


def interactive_prediction(model, vectorizer) -> None:
    """
    Prompts the user to enter email messages for real-time classification.
    Continues until the user types 'quit' or 'exit'.
    """
    print("\n" + "=" * 60)
    print("          CUSTOM EMAIL SPAM DETECTOR")
    print("=" * 60)
    print("Type any email message to check if it is Spam or Ham.")
    print("Type  'quit'  or  'exit'  to stop.\n")

    while True:
        user_input = input("Enter email message: ").strip()

        if not user_input:
            print("  [!] No message entered — please try again.\n")
            continue

        if user_input.lower() in ("quit", "exit"):
            print("\n[✔] Exiting spam classifier. Goodbye!")
            break

        result = predict_message(model, vectorizer, user_input)

        print("\n" + "-" * 45)
        if result == "Spam":
            print(f"  Result  :  {result}")
            print("  This message looks like SPAM — be careful!")
        else:
            print(f"  Result  :  {result}")
            print("  This message looks SAFE (Not Spam).")
        print("-" * 45 + "\n")

In [ ]:
def main():
    # Step 1 & 2 — Load and display the dataset
    df = load_data()

    # Step 3 — Clean the text
    df = preprocess_dataset(df)

    # Step 4 — Encode labels (Ham=0, Spam=1)
    X, y = encode_labels(df)

    # Step 5 — Train/test split
    X_train, X_test, y_train, y_test = split_dataset(X, y)

    # Step 6 — TF-IDF vectorization
    vectorizer, X_train_tfidf, X_test_tfidf = vectorize_text(X_train, X_test)

    # Step 7 — Train the Naive Bayes model
    model = train_model(X_train_tfidf, y_train)

    # Step 8 — Evaluate performance
    evaluate_model(model, X_test_tfidf, y_test)

    # Step 9 — Interactive prediction loop
    interactive_prediction(model, vectorizer)

In [ ]:
if __name__ == "__main__":
    main()

          EMAIL SPAM CLASSIFIER

[✔] Dataset loaded  —  50 records

First 5 rows:
                                                                     Message Label
Congratulations you have won a 1000 dollar gift card click here to claim now  Spam
                                 Hey are we still meeting for lunch tomorrow   Ham
     URGENT your account has been compromised verify now at fakebank dot com  Spam
                     Can you please send me the project report by end of day   Ham
                FREE iPhone limited time offer reply WIN to claim your prize  Spam

Label distribution:
Label
Spam    25
Ham     25

[✔] Text preprocessing complete.

Sample — original vs cleaned:
  Original : Congratulations you have won a 1000 dollar gift card click h
  Cleaned  : congratulations you have won a dollar gift card click here t

  Original : Hey are we still meeting for lunch tomorrow
  Cleaned  : hey are we still meeting for lunch tomorrow

  Original : URGENT your account has been 